# W7D3 — Speech-to-Text with Whisper — Guided

**Week 7 · Day 3 · Vision, language and speech** · Lab

<a href="https://colab.research.google.com/github/0xRush/AIEP_Olo_student/blob/master/week_7_multimodal/labs/D3_speech_to_text/D3_speech_to_text_guided.ipynb" target="_blank">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **The big idea.** Sound is a list of numbers — 16,000 of them per second. Whisper turns that list
> into a **log-mel spectrogram** (a picture of which frequencies are loud at each moment), reads the
> picture with a transformer **encoder**, and writes the words with a transformer **decoder** — the
> same encoder–decoder you saw for translation in week 5.

You will transcribe real speech in English and Arabic, build the metric everyone uses to score it
(**word error rate**), and then break the model on purpose: the wrong sample rate, noise, silence,
and audio longer than it can see at once.

<div dir="rtl" align="right">

# W7D3 — تحويل الكلام إلى نص مع Whisper

**الأسبوع السابع · اليوم الثالث · الرؤية واللغة والكلام** · معمل

<a href="https://colab.research.google.com/github/0xRush/AIEP_Olo_student/blob/master/week_7_multimodal/labs/D3_speech_to_text/D3_speech_to_text_guided.ipynb" target="_blank">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **الفكرة الكبرى.** الصوت قائمة أعداد — ستة عشر ألفًا منها في الثانية. يحوّل Whisper تلك القائمة إلى **مخطّط
> طيفي لوغاريتمي على مقياس ميل** (log-mel spectrogram، صورة لأيّ الترددات عالٍ في كل لحظة)، ويقرأ الصورة
> بـ**مُرمِّز** محوّل، ويكتب الكلمات بـ**مُفكِّك** محوّل — المُرمِّز والمُفكِّك نفسيهما اللذين رأيتهما للترجمة في الأسبوع
> الخامس.

ستُفرِّغ كلامًا حقيقيًا بالإنجليزية والعربية، وتبني المقياس الذي يستعمله الجميع لتقييمه (**معدّل خطأ الكلمات**)،
ثم تكسر النموذج عن قصد: بمعدّل عيّنات خاطئ، وبالضجيج، وبالصمت، وبصوت أطول مما يستطيع أن يراه دفعة واحدة.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

- Load audio, listen to it, and read its sample rate — and fix a wrong one by resampling.
- Compute Whisper's log-mel spectrogram and explain its `(80, 3000)` shape.
- Implement **word error rate** from the edit distance, and check it against `jiwer`.
- See how much of a "bad" WER is only punctuation and capital letters.
- Compare two model sizes on accuracy and speed (the **real-time factor**).
- Measure Whisper on Arabic, and on noisy, silent and long audio.

<div dir="rtl" align="right">

## أهداف التعلّم

- أن تُحمّل الصوت وتستمع إليه وتقرأ معدّل عيّناته — وتُصلح المعدّل الخاطئ بإعادة التشكيل.
- أن تحسب المخطّط الطيفي لـWhisper وتشرح شكله `(80, 3000)`.
- أن تنفّذ **معدّل خطأ الكلمات** من مسافة التحرير، وتتحقّق منه مقابل `jiwer`.
- أن ترى كم من معدّل الخطأ «السيّئ» ليس إلا علامات ترقيم وأحرفًا كبيرة.
- أن تقارن حجمين للنموذج في الدقّة والسرعة (**معامل الزمن الحقيقي**).
- أن تقيس Whisper على العربية، وعلى الصوت المشوَّش والصامت والطويل.

</div>

## About the models and data

**Models** (downloaded once, then cached):

| model | parameters | size |
|---|---|---|
| `openai/whisper-tiny` | 39 M | 151 MB |
| `openai/whisper-base` | 74 M | 290 MB |

Both are multilingual: the same checkpoint transcribes English and Arabic.

**Data:** `speech_clips_small` (10.5 MB) — real read speech with the correct transcript of every clip.

- `en/` — 30 English clips from LibriSpeech, one per speaker, 3–10 s each.
- `ar/` — 20 Modern Standard Arabic clips from FLEURS, read with an Egyptian accent, 5–12 s each.
- `long/en_long.wav` — 58 s of one speaker, for the 30-second-window task.
- `en_44k.wav` — the first English clip recorded at 44.1 kHz, like a phone or a laptop microphone.

All of it runs on a laptop CPU: the whole notebook takes about two minutes of computing.

<div dir="rtl" align="right">

## عن النماذج والبيانات

**النماذج** (تُنزَّل مرّة ثم تُحفظ):

| النموذج | المعاملات | الحجم |
|---|---|---|
| `openai/whisper-tiny` | ‏٣٩ مليونًا | ‏١٥١ ميجابايت |
| `openai/whisper-base` | ‏٧٤ مليونًا | ‏٢٩٠ ميجابايت |

كلاهما متعدّد اللغات: النسخة نفسها تُفرِّغ الإنجليزية والعربية.

**البيانات:** `speech_clips_small` (‏١٠٫٥ ميجابايت) — كلام مقروء حقيقي مع النص الصحيح لكل مقطع.

- `en/` — ‏٣٠ مقطعًا إنجليزيًا من LibriSpeech، مقطع لكل متحدّث، من ٣ إلى ١٠ ثوانٍ.
- `ar/` — ‏٢٠ مقطعًا بالعربية الفصحى من FLEURS، مقروءة بلهجة مصرية، من ٥ إلى ١٢ ثانية.
- `long/en_long.wav` — ‏٥٨ ثانية لمتحدّث واحد، لمهمة نافذة الثلاثين ثانية.
- `en_44k.wav` — المقطع الإنجليزي الأول مسجّلًا بـ٤٤٫١ كيلوهرتز، كما يسجّل الهاتف أو ميكروفون الحاسوب.

كله يعمل على معالج حاسوب محمول: يستغرق الدفتر كله نحو دقيقتين من الحساب.

</div>

## Setup

Run this first. On Colab it installs the course package; locally it uses your checkout.

<div dir="rtl" align="right">

## الإعداد

شغّل هذه الخلية أولًا. على Colab تُثبّت حزمة المقرّر، ومحلّيًا تستعمل نسختك من المستودع.

</div>

In [ ]:
# === AIEP portable setup — works locally and on Google Colab ========================
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
ensure('transformers', 'torch', 'soundfile', 'jiwer', 'scipy', 'matplotlib', 'pandas')
seed_everything(42)

from aiep.data import get_dataset_dir
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

import re
import time

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import matplotlib.pyplot as plt
import transformers
from IPython.display import Audio, display
from scipy.signal import resample_poly
from transformers import WhisperForConditionalGeneration, WhisperProcessor

torch.set_grad_enabled(False)              # nothing is trained today
transformers.logging.set_verbosity_error()  # Whisper's generate() is chatty
SEED, RATE = 42, 16_000                     # Whisper hears 16,000 samples per second

ROOT = get_dataset_dir("speech_clips_small")
CLIPS = pd.read_csv(ROOT / "transcripts.csv")
EN = CLIPS[CLIPS.file.str.startswith("en/")].reset_index(drop=True)
AR = CLIPS[CLIPS.file.str.startswith("ar/")].reset_index(drop=True)


def load(file):
    """A clip as float32 samples in [-1, 1], and its sample rate."""
    audio, rate = sf.read(ROOT / file, dtype="float32")
    return audio, rate


print(f"{len(EN)} English clips ({EN.seconds.sum():.0f} s), {len(AR)} Arabic clips ({AR.seconds.sum():.0f} s)")
print(versions(), "| device:", device())

## Section 1 — Warm-up: hear it, see it, transcribe it  *(~10 min)*

Nothing to write. Listen to one clip, look at its waveform, and let Whisper-tiny write it down. The
`transcribe` helper below is what the rest of the lab uses.

<div dir="rtl" align="right">

## القسم ١ — إحماء: اسمعه، وانظر إليه، وفرِّغه  *(≈١٠ دقائق)*

لا شيء تكتبه. استمع إلى مقطع، وانظر إلى موجته، ودع Whisper-tiny يكتبه. والدالة المساعدة `transcribe` أدناه
هي ما يستعمله باقي المعمل.

</div>

In [ ]:
audio, rate = load(EN.file[0])
print(f"{len(audio):,} samples at {rate:,} Hz = {len(audio) / rate:.2f} s")
display(Audio(audio, rate=rate))

plt.figure(figsize=(10, 2.5))
plt.plot(np.arange(len(audio)) / rate, audio, lw=0.5)
plt.xlabel("seconds"); plt.ylabel("amplitude"); plt.title(f"reference: {EN.text[0]!r}")
plt.show()

In [ ]:
tiny_processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
tiny = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").eval()


def transcribe(audio, model=tiny, processor=tiny_processor, language="english"):
    """Whisper's transcript of 16 kHz audio. language=None lets the model detect it."""
    features = processor(audio, sampling_rate=RATE, return_tensors="pt").input_features
    ids = model.generate(features, language=language, task="transcribe")
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()


print("reference :", EN.text[0])
print("whisper   :", transcribe(audio))

## Section 2 — From waveform to text  *(~55 min)*

Five tasks:

1. **Sample rate** — a 44.1 kHz recording read as if it were 16 kHz, and the fix.
2. **The spectrogram** — what Whisper's encoder actually sees.
3. **`wer()`** — the word error rate, from scratch.
4. **WER on 30 clips** — and how much of it is only punctuation.
5. **tiny vs base** — accuracy against speed.

<div dir="rtl" align="right">

## القسم ٢ — من الموجة إلى النص  *(≈٥٥ دقيقة)*

خمس مهام:

١. **معدّل العيّنات** — تسجيل بـ٤٤٫١ كيلوهرتز يُقرأ كأنه ١٦ كيلوهرتز، والإصلاح.
٢. **المخطّط الطيفي** — ما يراه مُرمِّز Whisper فعلًا.
٣. **`wer()`** — معدّل خطأ الكلمات، من الصفر.
٤. **معدّل الخطأ على ٣٠ مقطعًا** — وكم منه ليس إلا علامات ترقيم.
٥. **tiny مقابل base** — الدقّة مقابل السرعة.

</div>

### Task 1 — The sample rate

A sample rate is how many numbers make one second. `en_44k.wav` holds the same sentence as the clip
above, recorded at 44,100 per second. Whisper assumes 16,000 — so given the raw file it hears a voice
2.76 times slower and lower. Hear what it writes, then **resample** to 16 kHz and try again.

`resample_poly(audio, up, down)` changes the rate by `up / down`: 16,000 / 44,100 = 160 / 441.

<div dir="rtl" align="right">

### المهمّة ١ — معدّل العيّنات

معدّل العيّنات هو عدد الأعداد التي تصنع ثانية واحدة. يحمل `en_44k.wav` الجملة نفسها التي في المقطع أعلاه،
مسجّلةً بـ٤٤٬١٠٠ عيّنة في الثانية. ويفترض Whisper ١٦٬٠٠٠ — فإن أُعطي الملف الخام سمع صوتًا أبطأ وأخفض بـ٢٫٧٦
مرّة. اسمع ما يكتبه، ثم **أعِد التشكيل** إلى ١٦ كيلوهرتز وجرّب مرّة أخرى.

`resample_poly(audio, up, down)` يغيّر المعدّل بنسبة `up / down`: ‏16,000 / 44,100 = 160 / 441.

</div>

In [ ]:
audio_44k, rate_44k = load("en_44k.wav")
print(f"en_44k.wav: {rate_44k:,} Hz, {len(audio_44k):,} samples")
WRONG = transcribe(audio_44k)              # 44.1 kHz numbers, read as 16 kHz
print("as if 16 kHz :", WRONG)

# TODO: Resample audio_44k to 16 kHz into audio_16k, then transcribe it into FIXED.
# مهمة: أعِد تشكيل `audio_44k` إلى ١٦ كيلوهرتز في `audio_16k`، ثم فرِّغه في `FIXED`.
print("resampled    :", FIXED)
print(f"{len(audio_16k):,} samples now = {len(audio_16k) / RATE:.2f} s")
display(Audio(audio_44k, rate=RATE))       # what Whisper heard the first time

### Task 2 — What the encoder sees

Whisper never reads samples directly. Its processor turns audio into a **log-mel spectrogram**: 80
rows (frequency bands, low at the bottom) by one column every 10 ms. It always makes exactly
**30 seconds** of columns — shorter audio is padded with silence. Compute it for the first clip and
look at it: the bright stripes are the voice.

<div dir="rtl" align="right">

### المهمّة ٢ — ما يراه المُرمِّز

لا يقرأ Whisper العيّنات مباشرةً أبدًا. يحوّل معالجُه الصوت إلى **مخطّط طيفي لوغاريتمي على مقياس ميل**: ‏٨٠ صفًّا
(نطاقات ترددية، المنخفض في الأسفل) في عمود كل ١٠ مللي ثانية. ويصنع دائمًا أعمدةً لـ**٣٠ ثانية** بالضبط — والصوت
الأقصر يُحشى بالصمت. احسبه للمقطع الأول وانظر إليه: الخطوط الساطعة هي الصوت.

</div>

In [ ]:
# TODO: Compute the log-mel spectrogram of `audio` (the first clip) into MEL, shape (80, 3000).
# مهمة: احسب المخطّط الطيفي لـ`audio` (المقطع الأول) في `MEL`، بالشكل `(80, 3000)`.
print("spectrogram:", MEL.shape, f"= 80 frequency bands x {MEL.shape[1]} frames of 10 ms = 30 s")
print(f"the clip itself is {len(audio) / RATE:.1f} s; the rest is padding")

plt.figure(figsize=(11, 3))
plt.imshow(MEL, origin="lower", aspect="auto", cmap="magma", extent=[0, 30, 0, 80])
plt.axvline(len(audio) / RATE, color="white", ls="--", lw=1)
plt.xlabel("seconds"); plt.ylabel("mel band"); plt.title("what Whisper's encoder sees")
plt.show()

### Task 3 — Word error rate

**WER** counts the fewest word edits that turn the transcript into the reference — substitutions,
deletions and insertions — divided by the number of reference words. It is the **edit distance**
on words, the same dynamic programme as for letters:

`d[i][j]` = edits to turn the first `i` reference words into the first `j` hypothesis words:
`d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + (0 if the words match else 1))`.

WER can go above 1: a model that invents words makes insertions the reference never had.

<div dir="rtl" align="right">

### المهمّة ٣ — معدّل خطأ الكلمات

يعدّ **WER** أقلّ عدد من تعديلات الكلمات يحوّل النص المُفرَّغ إلى المرجع — الاستبدالات والحذوف والإدراجات —
مقسومًا على عدد كلمات المرجع. إنه **مسافة التحرير** على الكلمات، البرمجة الديناميكية نفسها التي للحروف:

`d[i][j]` = التعديلات التي تحوّل أول `i` كلمات من المرجع إلى أول `j` كلمات من الفرضية:
`d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + (0 إن تطابقت الكلمتان وإلا 1))`.

وقد يتجاوز WER الواحد: فالنموذج الذي يخترع كلمات يُدرج ما لم يكن في المرجع قط.

</div>

In [ ]:
import jiwer


def wer(reference, hypothesis):
    """Word error rate between two strings, by word-level edit distance."""
    ref, hyp = reference.split(), hypothesis.split()
    d = np.zeros((len(ref) + 1, len(hyp) + 1), dtype=int)
    d[:, 0], d[0, :] = np.arange(len(ref) + 1), np.arange(len(hyp) + 1)
    # TODO: Fill d with the edit-distance formula, then return d[-1, -1] / len(ref).
    # مهمة: املأ `d` بصيغة مسافة التحرير، ثم أعِد `d[-1, -1] / len(ref)`.


PAIRS = [("the cat sat on the mat", "the cat sat on the mat"),     # perfect
         ("the cat sat on the mat", "the cat sat on a mat"),       # 1 substitution of 6
         ("the cat sat on the mat", "the cat sat"),                 # 3 deletions of 6
         ("the cat sat", "oh the cat sat down there")]             # 3 insertions of 3
for ref, hyp in PAIRS:
    print(f"{wer(ref, hyp):.3f}  jiwer {jiwer.wer(ref, hyp):.3f}   {hyp!r}")

### Task 4 — WER on 30 English clips

Transcribe all 30 English clips with Whisper-tiny and score them. The references are lower-case with
no punctuation; Whisper writes `"Hello, world."`. Score twice: on the raw text, then after
`normalise_en` (lower-case, punctuation removed). The difference is not the model's fault.

<div dir="rtl" align="right">

### المهمّة ٤ — معدّل الخطأ على ٣٠ مقطعًا إنجليزيًا

فرِّغ المقاطع الإنجليزية الثلاثين كلها بـWhisper-tiny وقيّمها. المراجع بأحرف صغيرة بلا ترقيم، ويكتب Whisper
`"Hello, world."`. قيّم مرّتين: على النص الخام، ثم بعد `normalise_en` (أحرف صغيرة، وحذف الترقيم). والفرق ليس
ذنب النموذج.

</div>

In [ ]:
def normalise_en(text):
    """Lower-case, and keep only letters, apostrophes and spaces."""
    return " ".join(re.sub(r"[^a-z' ]", " ", text.lower()).split())


started = time.perf_counter()
# TODO: Transcribe every English clip with tiny into TINY_HYPS, then compute RAW_WER and NORM_WER over all clips.
# مهمة: فرِّغ كل مقطع إنجليزي بـtiny في `TINY_HYPS`، ثم احسب `RAW_WER` و`NORM_WER` على المقاطع كلها.
TINY_SECONDS = time.perf_counter() - started
print(f"raw WER {RAW_WER:.3f}  |  normalised WER {NORM_WER:.3f}   ({TINY_SECONDS:.0f} s)")
for ref, hyp in list(zip(EN.text, TINY_HYPS))[:4]:
    print(f"  ref: {ref}\n  hyp: {hyp}\n")

### Task 5 — tiny vs base

A bigger model should make fewer errors, and take longer. Measure both. The **real-time factor**
(RTF) is seconds of computing per second of audio: below 1 means faster than real time.

<div dir="rtl" align="right">

### المهمّة ٥ — tiny مقابل base

يُفترض أن يُخطئ النموذج الأكبر أقلّ، وأن يستغرق أطول. قِس الاثنين. **معامل الزمن الحقيقي** (RTF) هو ثواني الحساب
لكل ثانية من الصوت: أقلّ من ١ يعني أسرع من الزمن الحقيقي.

</div>

In [ ]:
base_processor = WhisperProcessor.from_pretrained("openai/whisper-base")
base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base").eval()

started = time.perf_counter()
# TODO: Transcribe every English clip with base into BASE_HYPS and compute its normalised WER, BASE_WER.
# مهمة: فرِّغ كل مقطع إنجليزي بـbase في `BASE_HYPS` واحسب معدّل خطئه الموحَّد `BASE_WER`.
BASE_SECONDS = time.perf_counter() - started

AUDIO_SECONDS = EN.seconds.sum()
SIZES = pd.DataFrame([("whisper-tiny", 39, NORM_WER, TINY_SECONDS / AUDIO_SECONDS),
                      ("whisper-base", 74, BASE_WER, BASE_SECONDS / AUDIO_SECONDS)],
                     columns=["model", "M params", "WER", "real-time factor"])
print(SIZES.round(3).to_string(index=False))

### Takeaway

Before you trust any WER, check what was compared with what. Half of tiny's raw errors were commas
and capital letters — a normalisation choice, not a hearing problem.

<div dir="rtl" align="right">

### الخلاصة

قبل أن تثق بأي معدّل خطأ، تحقّق ممّا قورن بماذا. فنصف أخطاء tiny الخام كانت فواصل وأحرفًا كبيرة — اختيارًا في
التوحيد، لا مشكلة في السمع.

</div>

## Section 3 — Where it breaks  *(~35 min)*

Three tasks:

1. **Arabic** — and how much of the error is spelling, not hearing.
2. **Noise** — WER as the signal gets buried.
3. **Silence and long audio** — Whisper invents words, and stops at 30 seconds.

<div dir="rtl" align="right">

## القسم ٣ — أين ينكسر  *(≈٣٥ دقيقة)*

ثلاث مهام:

١. **العربية** — وكم من الخطأ إملاءٌ لا سمع.
٢. **الضجيج** — معدّل الخطأ حين تُدفن الإشارة.
٣. **الصمت والصوت الطويل** — يخترع Whisper كلمات، ويتوقّف عند ٣٠ ثانية.

</div>

### Task 1 — Arabic

Transcribe the 20 Arabic clips with Whisper-base (`language="arabic"`). Whisper-tiny is not worth
trying here — on these clips it scores a WER of 1.28: more wrong words than there are words.

Then re-score with week 5's `normalise_arabic` applied to **both** sides. The reference and Whisper
often spell the same spoken word differently — `إ` or `ا`, `ة` or `ه` — and WER counts every one as
an error. Also ask the model to detect the language itself (`language=None`) on one clip.

<div dir="rtl" align="right">

### المهمّة ١ — العربية

فرِّغ المقاطع العربية العشرين بـWhisper-base ‏(`language="arabic"`). ولا يستحق Whisper-tiny التجربة هنا — فعلى
هذه المقاطع يبلغ معدّل خطئه ١٫٢٨: كلمات خاطئة أكثر من عدد الكلمات.

ثم أعِد التقييم بعد تطبيق `normalise_arabic` من الأسبوع الخامس على **الطرفين** كليهما. فكثيرًا ما يكتب المرجع
وWhisper الكلمة المنطوقة نفسها كتابتين مختلفتين — `إ` أو `ا`، و`ة` أو `ه` — ويعدّ WER كلًّا منها خطأً. واطلب
أيضًا من النموذج أن يكتشف اللغة بنفسه (`language=None`) على مقطع واحد.

</div>

In [ ]:
def normalise_arabic(text):
    """Week 5's normaliser, plus punctuation removed: diacritics, alef forms, taa marbuta, alef maqsura."""
    text = re.sub(r"[ً-ْٰـ]", "", text)
    text = re.sub(r"[أإآٱ]", "ا", text)
    text = text.replace("ة", "ه").replace("ى", "ي")
    return " ".join(re.sub(r"[^\w\s]", " ", text).split())


started = time.perf_counter()
# TODO: Transcribe the Arabic clips with base into AR_HYPS, then compute AR_RAW_WER and AR_NORM_WER.
# مهمة: فرِّغ المقاطع العربية بـbase في `AR_HYPS`، ثم احسب `AR_RAW_WER` و`AR_NORM_WER`.
print(f"Arabic, whisper-base: raw WER {AR_RAW_WER:.3f} | normalised {AR_NORM_WER:.3f} "
      f"({time.perf_counter() - started:.0f} s)")
print(f"  ref: {AR.text[0]}\n  hyp: {AR_HYPS[0]}")
print("\nlanguage detected by itself:", transcribe(load(AR.file[0])[0], base, base_processor, language=None))

### Task 2 — Noise

Add white noise at a chosen **signal-to-noise ratio** (SNR, in decibels): 20 dB is a quiet room,
10 dB a busy café, 0 dB means the noise is as loud as the voice. Noise power is the signal's power
divided by `10 ** (snr / 10)`. Score Whisper-tiny on 15 clips at each level.

<div dir="rtl" align="right">

### المهمّة ٢ — الضجيج

أضِف ضجيجًا أبيض عند **نسبة إشارة إلى ضجيج** مختارة (SNR، بالديسيبل): ‏٢٠ ديسيبل غرفة هادئة، و١٠ ديسيبل مقهى
مزدحم، و٠ ديسيبل يعني أن الضجيج بارتفاع الصوت نفسه. وقدرة الضجيج هي قدرة الإشارة مقسومةً على `10 ** (snr / 10)`.
قيّم Whisper-tiny على ١٥ مقطعًا عند كل مستوى.

</div>

In [ ]:
rng = np.random.default_rng(SEED)


def add_noise(audio, snr_db):
    """`audio` plus white noise at `snr_db` decibels below it."""
    # TODO: Scale unit-power white noise to the right power for snr_db and return audio + noise (float32).
    # مهمة: اضبط ضجيجًا أبيض قدرته ١ إلى القدرة المناسبة لـ`snr_db` وأعِد `audio + noise` بنوع float32.


NOISY = EN.head(15)
NOISE_WER = {}
for snr in [20, 10, 0]:
    hyps = [transcribe(add_noise(load(f)[0], snr)) for f in NOISY.file]
    NOISE_WER[snr] = wer(" ".join(map(normalise_en, NOISY.text)), " ".join(map(normalise_en, hyps)))
    print(f"SNR {snr:>2} dB: WER {NOISE_WER[snr]:.3f}")
display(Audio(add_noise(load(EN.file[0])[0], 0), rate=RATE))
plt.plot(list(NOISE_WER), list(NOISE_WER.values()), "o-"); plt.gca().invert_xaxis()
plt.xlabel("SNR (dB) — noisier to the right"); plt.ylabel("WER"); plt.show()

### Task 3 — Silence, and audio longer than 30 seconds

Two failures every speech product has to handle:

- **Silence.** Feed Whisper five seconds of nothing, then five seconds of faint noise. It was trained
  on subtitles, so it has a habit of writing what subtitles say at the end of a video.
- **Long audio.** The encoder only ever sees a 30-second window. Give it the 58-second clip and count
  the words that come back. Then use the `pipeline` with `chunk_length_s=30`, which cuts the audio
  into windows and joins the transcripts.

<div dir="rtl" align="right">

### المهمّة ٣ — الصمت، والصوت الأطول من ٣٠ ثانية

إخفاقان على كل منتج كلامي أن يتعامل معهما:

- **الصمت.** أطعِم Whisper خمس ثوانٍ من لا شيء، ثم خمس ثوانٍ من ضجيج خافت. دُرّب على ترجمات الأفلام المكتوبة،
  فله عادة أن يكتب ما تقوله الترجمات في آخر الفيديو.
- **الصوت الطويل.** لا يرى المُرمِّز إلا نافذة من ٣٠ ثانية. أعطِه المقطع ذا الثماني والخمسين ثانية وعُدّ الكلمات
  التي تعود. ثم استعمل `pipeline` مع `chunk_length_s=30`، الذي يقطع الصوت إلى نوافذ ويصل النصوص.

</div>

In [ ]:
SILENCE = transcribe(np.zeros(5 * RATE, dtype="float32"))
FAINT = transcribe(rng.normal(0, 0.003, 5 * RATE).astype("float32"))
print(f"5 s of silence     -> {SILENCE!r}")
print(f"5 s of faint noise -> {FAINT!r}")

In [ ]:
from transformers import pipeline

long_audio, _ = load("long/en_long.wav")
LONG_REF = CLIPS.loc[CLIPS.file == "long/en_long.wav", "text"].iloc[0]
# TODO: Transcribe long_audio once plainly into WHOLE, and once with a chunking pipeline into CHUNKED.
# مهمة: فرِّغ `long_audio` مرّةً بالنداء العادي في `WHOLE`، ومرّةً بخطّ معالجة يقطّع الصوت في `CHUNKED`.
print(f"{len(long_audio) / RATE:.0f} s of audio, {len(LONG_REF.split())} words in the reference")
print(f"plain call  : {len(WHOLE.split())} words — ends with ...{WHOLE[-60:]!r}")
print(f"chunked     : {len(CHUNKED.split())} words, WER {wer(normalise_en(LONG_REF), normalise_en(CHUNKED)):.3f}")

### Write-up

In **one sentence**: which of today's failures — wrong sample rate, Arabic, noise, silence, or long
audio — would hurt a real product most, and how would you guard against it?

> *Example: "Silence, because a voice assistant that hears an empty room and writes 'Thank you' will
> act on words nobody said — so put a voice-activity detector in front and never send silent audio to
> the model."*

<div dir="rtl" align="right">

### كتابة

في **جملة واحدة**: أيّ إخفاقات اليوم — معدّل العيّنات الخاطئ، أو العربية، أو الضجيج، أو الصمت، أو الصوت الطويل —
يضرّ منتجًا حقيقيًا أكثر، وكيف تحتاط منه؟

> *مثال: «الصمت، لأن مساعدًا صوتيًا يسمع غرفة فارغة فيكتب "شكرًا لك" سيتصرّف بكلمات لم يقلها أحد — فضع أمامه
> كاشفًا للنشاط الصوتي ولا ترسل صوتًا صامتًا إلى النموذج أبدًا.»*

</div>

In [ ]:
YOUR_ANSWER = """
(write your sentence here)
"""

In [ ]:
pd.DataFrame({"model": ["whisper-tiny", "whisper-base", "whisper-base"], "language": ["en", "en", "ar"],
              "wer": [NORM_WER, BASE_WER, AR_NORM_WER]}).to_csv(ARTEFACT_DIR / "whisper_wer.csv", index=False)
print("wrote whisper_wer.csv — tomorrow's text-to-speech lab scores its voices with the same wer()")

## Wrap-up

- Audio is samples at a sample rate; the wrong rate is silent until the transcript turns to nonsense.
- Whisper = log-mel spectrogram (30 s, 80 bands) → transformer encoder → decoder that writes text.
- WER is the word edit distance over the reference length — and it must be computed on normalised
  text, in Arabic as much as in English.
- Real audio brings noise, silence and length; each one needs handling outside the model.

Tomorrow (**D4**) the arrow turns round: text goes in and speech comes out — and today's Whisper and
`wer()` become the judge of how understandable that speech is.

<div dir="rtl" align="right">

## الخلاصة

- الصوت عيّنات بمعدّل عيّنات؛ والمعدّل الخاطئ صامت حتى يتحوّل النص إلى هراء.
- Whisper = مخطّط طيفي (‏٣٠ ثانية، ٨٠ نطاقًا) ← مُرمِّز محوّل ← مُفكِّك يكتب النص.
- WER هو مسافة تحرير الكلمات مقسومةً على طول المرجع — ويجب أن يُحسب على نصّ موحَّد، في العربية كما في الإنجليزية.
- الصوت الحقيقي يأتي بالضجيج والصمت والطول؛ وكلٌّ منها يحتاج إلى معالجة خارج النموذج.

وغدًا (**اليوم ٤**) ينقلب السهم: يدخل النص ويخرج الكلام — ويصير Whisper اليوم و`wer()` حَكَمًا على مدى وضوح ذلك
الكلام.

</div>

In [ ]:
check(abs(len(audio_16k) - len(audio_44k) * 16000 / 44100) < 2 and NORM_WER < 0.2 and normalise_en(FIXED) != normalise_en(WRONG),
      f"resampling must give 16 kHz audio and change the transcript — got {len(audio_16k)} samples, {FIXED!r}",
      f"يجب أن تُعطي إعادة التشكيل صوتًا بـ١٦ كيلوهرتز وتُغيّر النص — والناتج {len(audio_16k)} عيّنة، {FIXED!r}")
check(MEL.shape == (80, 3000),
      f"Whisper's log-mel spectrogram must be (80, 3000) — got {MEL.shape}",
      f"يجب أن يكون المخطّط الطيفي لـWhisper بالشكل `(80, 3000)` — والناتج {MEL.shape}")
check(all(abs(wer(r, h) - jiwer.wer(r, h)) < 1e-9 for r, h in PAIRS) and wer(*PAIRS[3]) == 1.0,
      f"wer() must agree with jiwer on all four pairs, and 3 insertions on 3 words must give 1.0 — got "
      f"{[round(wer(r, h), 3) for r, h in PAIRS]}",
      f"يجب أن تتّفق `wer()` مع jiwer على الأزواج الأربعة، وأن تُعطي ٣ إدراجات على ٣ كلمات القيمة ١٫٠ — والناتج "
      f"{[round(wer(r, h), 3) for r, h in PAIRS]}")
check(NORM_WER < RAW_WER and BASE_WER <= NORM_WER + 0.02,
      f"normalising must lower tiny's WER ({RAW_WER:.3f} -> {NORM_WER:.3f}), and base must be at least as good ({BASE_WER:.3f})",
      f"يجب أن يخفّض التوحيد معدّل خطأ tiny ‏({RAW_WER:.3f} -> {NORM_WER:.3f})، وأن يكون base جيّدًا مثله على الأقل ({BASE_WER:.3f})")
check(AR_NORM_WER < AR_RAW_WER < 1,
      f"normalising both sides must lower the Arabic WER — got {AR_RAW_WER:.3f} -> {AR_NORM_WER:.3f}",
      f"يجب أن يخفّض توحيد الطرفين معدّل الخطأ العربي — والناتج {AR_RAW_WER:.3f} -> {AR_NORM_WER:.3f}")
check(NOISE_WER[0] > NOISE_WER[20] and len(CHUNKED.split()) > 1.5 * len(WHOLE.split()),
      f"noise must raise the WER ({NOISE_WER}), and chunking must recover the words past 30 s "
      f"({len(WHOLE.split())} -> {len(CHUNKED.split())} words)",
      f"يجب أن يرفع الضجيج معدّل الخطأ ({NOISE_WER})، وأن يستعيد التقطيع الكلمات بعد ٣٠ ثانية "
      f"({len(WHOLE.split())} -> {len(CHUNKED.split())} كلمة)")
report()